# HydraY NNUE — training v1 su Colab (T4)

Runtime → Cambia tipo di runtime → **GPU (T4)**.

Prerequisito: carica su Google Drive il file dati interleaved
`hydray_v1_121M_interleaved.bin.zst` (121M posizioni, etichette rete shakedown).
Poi esegui le celle in ordine. Tempo previsto (40 superbatch ≈ 4B sample): **~2-4 h**.
Checkpoint intermedi ogni 10 superbatch: se la sessione muore, l'ultimo salvato resta usabile.

In [ ]:
!nvidia-smi
!nvcc --version || ls /usr/local/cuda/bin

In [ ]:
# Rust toolchain
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal
!~/.cargo/bin/cargo --version

In [ ]:
# Trainer HydraY (solo la cartella nnue/trainer serve)
!git clone --depth 1 --branch Claudio1.3.0 https://github.com/ThomasGhione/chess_engine
%cd chess_engine/nnue/trainer

In [ ]:
# Dati da Google Drive (adatta il path al tuo Drive)
from google.colab import drive
drive.mount('/content/drive')
import os, glob
src = glob.glob('/content/drive/MyDrive/hydray_v1_121M_interleaved.bin*')[0]
print('trovato:', src)
if src.endswith('.zst'):
    !apt-get -qq install -y zstd
    !zstd -d -f "{src}" -o /content/data.bin
else:
    !cp "{src}" /content/data.bin
print(os.path.getsize('/content/data.bin') / 32, 'posizioni')

In [ ]:
# Training v1: 40 superbatch (~4B sample, ~33 epoche sui 121M). Loss deve SCENDERE.
# In coda stampa i sanity eval (startpos ~ +20..50 cp, coppia mirror identica, ±donna enorme).
!PATH=$HOME/.cargo/bin:$PATH CUDA_PATH=/usr/local/cuda cargo run -r --bin trainer --features cuda -- /content/data.bin 40 hydray-v1

In [ ]:
# Verifica standalone del file quantizzato (stesso lettore che fa da spec al loader C++)
!ls checkpoints/
!PATH=$HOME/.cargo/bin:$PATH cargo run -r --bin sanity -- checkpoints/hydray-v1-40/quantised.bin

In [ ]:
# Scarica il checkpoint (quantised.bin ~400 KB)
from google.colab import files
files.download('checkpoints/hydray-v1-40/quantised.bin')